# Lab 1 — PCA y monitoreo estructural — Solución vía IA (solo docente)

Prompts canónicos probados y código mínimo que pasa autoevaluación (✅).
No comparar línea a línea con el notebook del alumno: importa que llegue a los mismos resultados.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_contexto_pca, verificar_carga, verificar_limpieza, verificar_columna,
    verificar_resumen, verificar_etiquetas, verificar_correlacion, verificar_escalado,
    verificar_varianza, verificar_proyeccion_2d, verificar_loadings,
    verificar_clasificacion_pca, verificar_kmeans, verificar_dbscan,
    verificar_comparativa_clustering, verificar, resumen_seccion,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, silhouette_score, adjusted_rand_score

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (Codespaces / labs/.venv).')


## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración y movimiento en tres ejes |
| Strain | με | Deformación del material (extensómetro) |
| Temp | °C | Temperatura ambiente / del sensor |
| **Condition Label** | **0 / 1 / 2** | **Estado estructural (normal / daño menor / severo)** |

Fuente: Ziya07 · 1 000 lecturas · PCA **no supervisado** (no usa la etiqueta para crear componentes).

Detalle ampliado: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Contexto PCA en monitoreo estructural

### 📘 Subpreguntas
- **1.a** ¿PCA es un método **supervisado** o **no supervisado**?
- **1.b** ¿Por qué es útil reducir dimensionalidad antes de visualizar o clasificar lecturas de sensores?

En obra, decenas de sensores generan datos redundantes; PCA condensa esa información en pocas componentes interpretables.


#### ✅ Respuesta sugerida

**1.a** No supervisado: PCA solo usa las features de sensores, no `Condition Label`.
**1.b** Reduce ruido y redundancia (ej. ejes de aceleración correlacionados) para visualizar estados y acelerar clasificadores.


### 🤖 Sección 1 — Contexto PCA

**Objetivo:** Confirmar que el método de reducción es PCA.

**Lista de tareas**
1. METODO_REDUCCION = 'pca'
2. print del método
3. ✅ autoevaluación 1

**Prompt sugerido** (copiar al asistente):

```text
Lab SHM (sensores estructura). Celda ### PEGA AQUÍ ###:
METODO_REDUCCION = "pca"
print(f"Método elegido: {METODO_REDUCCION}")
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
METODO_REDUCCION = "pca"
print(f"Método elegido: {METODO_REDUCCION}")


In [ ]:
# --- Autoevaluación 1 ---
r = verificar_contexto_pca(METODO_REDUCCION)
resumen_seccion('1 — Contexto PCA', r)


## Pregunta 2 — Carga del dataset de sensores

### 📘 Subpreguntas
- **2.a** ¿Cuántas **features** de sensor hay frente a 1 **etiqueta** de condición?
- **2.b** ¿Por qué incluimos `Timestamp` pero no la usamos en PCA?


#### ✅ Reflexión 2

5 features + Timestamp + Condition Label. Timestamp ordena series temporales pero no entra al PCA.


In [ ]:
# --- PRE-ESCRITO: carga desde data/building_health_monitoring_dataset.csv ---
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")


### 🤖 Sección 2 — Carga

**Objetivo:** Definir 5 features de sensor y mostrar head del CSV.

**Lista de tareas**
1. FEATURES con 5 nombres exactos del CSV
2. N_FILAS_HEAD 1-20
3. display head

**Prompt sugerido** (copiar al asistente):

```text
df ya cargado desde data/building_health_monitoring_dataset.csv (1000×7).
Columnas sensor exactas: "Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)", "Strain (με)", "Temp (°C)"
Celda ### PEGA AQUÍ ###: lista FEATURES (5), N_FILAS_HEAD=5, print y display(df.head)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
FEATURES = [
    "Accel_X (m/s^2)",
    "Accel_Y (m/s^2)",
    "Accel_Z (m/s^2)",
    "Strain (με)",
    "Temp (°C)",
]
N_FILAS_HEAD = 5
print(f"Features para PCA: {FEATURES}")
print(f"Primeras {N_FILAS_HEAD} lecturas:")
display(df.head(N_FILAS_HEAD))


In [ ]:
# --- Autoevaluación 2 ---
r = verificar_carga(df, N_FILAS_HEAD)
r.append(verificar(len(FEATURES) == 5, '✅ 5 features de sensores definidas.', '❌ FEATURES debe incluir las 5 columnas de sensores.'))
resumen_seccion('2 — Carga', r)
print("ℹ️ 5 features + Timestamp + Condition Label.")


## Pregunta 3 — Calidad de datos y limpieza

### 📘 Subpreguntas
- **3.a** ¿Cuántas filas se pierden al eliminar lecturas con nulos?
- **3.b** ¿Qué sensor revisarías primero y por qué?


#### ✅ Reflexión 3

96 filas con nulos eliminadas → 904 lecturas. Strain es crítico por sensibilidad al daño.


In [ ]:
# --- PRE-ESCRITO: conteo de nulos y limpieza ---
n_antes = len(df)
n_nulos_por_col = df[FEATURES].isna().sum()
print("Nulos por sensor (datos crudos):")
display(n_nulos_por_col)

df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
print(f"Tras dropna: {n_antes} → {n_despues} lecturas válidas")


### 🤖 Sección 3 — Calidad

**Objetivo:** Revisar estadísticas de un sensor en datos crudos.

**Lista de tareas**
1. COLUMNA_REVISAR válida
2. describe y display

**Prompt sugerido** (copiar al asistente):

```text
df y df_limpio ya creados arriba. ### PEGA AQUÍ ###:
COLUMNA_REVISAR = "Strain (με)"
stats_col = df[COLUMNA_REVISAR].describe()
print y display
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
COLUMNA_REVISAR = "Strain (με)"
stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas de «{COLUMNA_REVISAR}» (datos crudos):")
display(stats_col)


In [ ]:
# --- Autoevaluación 3 ---
r = verificar_limpieza(df_limpio, n_antes, n_despues)
r.extend(verificar_columna(df, COLUMNA_REVISAR))
resumen_seccion('3 — Calidad', r)


## Pregunta 4 — Estadísticas descriptivas de sensores

### 📘 Subpreguntas
- **4.a** Mirando `describe()`, ¿qué sensor tiene mayor dispersión relativa (std vs |media|)?
- **4.b** ¿Por qué esto importa antes de aplicar PCA?


#### ✅ Reflexión 4

Sensores con magnitudes distintas (Accel_Z ~ 9.8 vs Strain ~ 80 με) exigen escalado antes de PCA.


### 🤖 Sección 4 — Describe

**Objetivo:** Resumen y dispersión relativa de sensores en df_limpio.

**Lista de tareas**
1. COLUMNAS_RESUMEN con ≥2 columnas válidas
2. describe y std/|media|

**Prompt sugerido** (copiar al asistente):

```text
Usar df_limpio. ### PEGA AQUÍ ###:
COLUMNAS_RESUMEN = ["Strain (με)", "Temp (°C)", "Accel_Z (m/s^2)"]
resumen = df_limpio[COLUMNAS_RESUMEN].describe(); display(resumen)
medias_abs y dispersion = std/medias_abs; display(dispersion)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
COLUMNAS_RESUMEN = ["Strain (με)", "Temp (°C)", "Accel_Z (m/s^2)"]
resumen = df_limpio[COLUMNAS_RESUMEN].describe()
display(resumen)
medias_abs = df_limpio[COLUMNAS_RESUMEN].abs().mean()
dispersion = (df_limpio[COLUMNAS_RESUMEN].std() / medias_abs).sort_values(ascending=False)
display(dispersion)


In [ ]:
# --- Autoevaluación 4 ---
r = verificar_resumen(resumen, COLUMNAS_RESUMEN)
resumen_seccion('4 — Describe', r)


## Pregunta 5 — Distribución de Condition Label

### 📘 Subpreguntas
- **5.a** ¿Qué clase es mayoritaria (0, 1 o 2)?
- **5.b** ¿Por qué es típico tener más lecturas «normales» en SHM?


#### ✅ Reflexión 5

Clase 0 mayoritaria (~637): la estructura pasa más tiempo en servicio normal.


In [ ]:
# --- PRE-ESCRITO: histograma de etiquetas ---
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(conteo.keys(), conteo.values(), color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
ax.set_xlabel('Condition Label')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de estados estructurales')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['0 — Normal', '1 — Daño menor', '2 — Daño severo'])
plt.tight_layout()
plt.show()


### 🤖 Sección 5 — Etiquetas

**Objetivo:** Mostrar conteo de Condition Label.

**Lista de tareas**
1. N_CLASES_MOSTRAR=3
2. usar dict conteo ya calculado

**Prompt sugerido** (copiar al asistente):

```text
conteo ya existe (df_limpio Condition Label). ### PEGA AQUÍ ###:
N_CLASES_MOSTRAR = 3
serie_clases = pd.Series(conteo).sort_index().head(N_CLASES_MOSTRAR)
print y display
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
N_CLASES_MOSTRAR = 3
serie_clases = pd.Series(conteo).sort_index().head(N_CLASES_MOSTRAR)
display(serie_clases)


In [ ]:
# --- Autoevaluación 5 ---
r = verificar_etiquetas(conteo, N_CLASES_MOSTRAR)
resumen_seccion('5 — Etiquetas', r)


## Pregunta 6 — Correlación entre sensores

### 📘 Subpreguntas
- **6.a** ¿Qué par de sensores está **más correlacionado** (|r|)?
- **6.b** ¿Qué implica alta correlación para PCA?


#### ✅ Reflexión 6

Alta correlación entre ejes de aceleración → redundancia que PCA puede condensar.


In [ ]:
# --- PRE-ESCRITO: matriz de correlación ---
corr = df_limpio[FEATURES].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlación entre sensores (datos limpios)')
plt.tight_layout()
plt.show()


### 🤖 Sección 6 — Correlación

**Objetivo:** Top correlaciones entre features en df_limpio.

**Lista de tareas**
1. TOP_N_CORR=3
2. matriz solo FEATURES

**Prompt sugerido** (copiar al asistente):

```text
df_limpio y FEATURES definidos. ### PEGA AQUÍ ###:
TOP_N_CORR = 3
corr = df_limpio[FEATURES].corr()
# imprimir pares con mayor |r| (sin diagonal)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
TOP_N_PARES = 3
pares = []
for i, c1 in enumerate(FEATURES):
    for c2 in FEATURES[i + 1:]:
        pares.append((c1, c2, abs(corr.loc[c1, c2])))
pares_ordenados = sorted(pares, key=lambda x: x[2], reverse=True)
top_pares = pares_ordenados[:TOP_N_PARES]
max_corr = top_pares[0][2]
top_par = (top_pares[0][0], top_pares[0][1])
for c1, c2, r_val in top_pares:
    print(f"  {c1} ↔ {c2}: |r| = {r_val:.3f}")


In [ ]:
# --- Autoevaluación 6 ---
r = verificar_correlacion(max_corr, top_par)
resumen_seccion('6 — Correlación', r)


## Pregunta 7 — Estandarización antes de PCA

### 📘 Subpreguntas
- **7.a** ¿Por qué `StandardScaler` es recomendable antes de PCA?
- **7.b** ¿Qué sensor domina PC1 si **no** escalas (pista: gravedad en Accel_Z)?


#### ✅ Reflexión 7

Sin escalado, Accel_Z domina PC1 (~gravedad). Con StandardScaler, PC1 ≈ 27% varianza.


In [ ]:
# --- PRE-ESCRITO: comparar varianza PC1 con y sin escalado ---
X_raw = df_limpio[FEATURES].values
scaler_ref = StandardScaler()
X_scaled_ref = scaler_ref.fit_transform(X_raw)

pca_crudo = PCA(n_components=5, random_state=42)
pca_crudo.fit(X_raw)
var_pc1_crudo = float(pca_crudo.explained_variance_ratio_[0])

pca_esc = PCA(n_components=5, random_state=42)
pca_esc.fit(X_scaled_ref)
var_pc1_escalado = float(pca_esc.explained_variance_ratio_[0])

print(f"PC1 sin escalado:  {var_pc1_crudo*100:.1f}% de varianza")
print(f"PC1 con escalado:  {var_pc1_escalado*100:.1f}% de varianza")


### 🤖 Sección 7 — Escalado

**Objetivo:** StandardScaler sobre X_features.

**Lista de tareas**
1. X_features desde df_limpio[FEATURES]
2. X_scaled con scaler.fit_transform

**Prompt sugerido** (copiar al asistente):

```text
### PEGA AQUÍ ###:
X_features = df_limpio[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)
print shape X_scaled
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
ESCALAR = True
if ESCALAR:
    scaler = StandardScaler()
    X_pca_input = scaler.fit_transform(X_raw)
else:
    X_pca_input = X_raw
pca_tmp = PCA(n_components=5, random_state=42)
pca_tmp.fit(X_pca_input)
var_pc1_actual = float(pca_tmp.explained_variance_ratio_[0])
print(f"ESCALAR={ESCALAR} → PC1 explica {var_pc1_actual*100:.1f}% de varianza")


In [ ]:
# --- Autoevaluación 7 ---
r = verificar_escalado(ESCALAR, var_pc1_escalado, var_pc1_crudo)
resumen_seccion('7 — Escalado', r)


## Pregunta 8 — Varianza explicada (scree plot)

### 📘 Subpreguntas
- **8.a** ¿Cuántas componentes necesitas para capturar el **90%** de varianza?
- **8.b** ¿Qué trade-off hay al usar menos componentes?


#### ✅ Reflexión 8

90% de varianza requiere las 5 componentes (todas las features originales).


In [ ]:
# --- PRE-ESCRITO: PCA completo y scree plot ---
pca_full = PCA(random_state=42)
pca_full.fit(X_pca_input)
var_ratio = pca_full.explained_variance_ratio_
var_acum = np.cumsum(var_ratio).tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(var_ratio) + 1), var_ratio, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada')
axes[0].set_title('Scree plot — varianza por componente')

axes[1].plot(range(1, len(var_acum) + 1), var_acum, 'o-', color='#e74c3c')
axes[1].axhline(0.90, color='gray', linestyle='--', label='90%')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada')
axes[1].set_title('Varianza acumulada')
axes[1].legend()
plt.tight_layout()
plt.show()


### 🤖 Sección 8 — Varianza PCA

**Objetivo:** Varianza explicada y N_COMPONENTES.

**Lista de tareas**
1. PCA con n_components=N_COMPONENTES (3-5)
2. var_acum lista

**Prompt sugerido** (copiar al asistente):

```text
X_scaled listo. ### PEGA AQUÍ ###:
N_COMPONENTES = 3
pca = PCA(n_components=N_COMPONENTES)
X_pca = pca.fit_transform(X_scaled)
var_acum = pca.explained_variance_ratio_.cumsum().tolist()
print varianza por componente
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
UMBRAL_VARIANZA = 0.90
N_COMPONENTES = 5
n_min = int(np.searchsorted(var_acum, UMBRAL_VARIANZA) + 1)
print(f"Umbral {UMBRAL_VARIANZA:.0%} → mínimo {n_min} componente(s)")
print(f"N_COMPONENTES = {N_COMPONENTES}")


In [ ]:
# --- Autoevaluación 8 ---
r = verificar_varianza(var_acum, UMBRAL_VARIANZA, N_COMPONENTES)
resumen_seccion('8 — Varianza', r)


## Pregunta 9 — Proyección 2D: PC1 vs PC2

### 📘 Subpreguntas
- **9.a** ¿Las clases de daño se **separan** visualmente en el plano PC1–PC2?
- **9.b** ¿Por qué coloreamos por `Condition Label` si PCA no la usó?


#### ✅ Reflexión 9

Colorear por etiqueta **interpreta** el gráfico; PCA no usó la etiqueta al calcular componentes.


In [ ]:
# --- PRE-ESCRITO: transformación a componentes principales ---
X_pca = pca_full.transform(X_pca_input)
var_pc1 = float(var_ratio[0])
var_pc2 = float(var_ratio[1])
print(f"Proyección lista: {X_pca.shape[0]} puntos × {X_pca.shape[1]} componentes")


### 🤖 Sección 9 — Proyección 2D

**Objetivo:** Scatter PC1 vs PC2 coloreado por Condition Label.

**Lista de tareas**
1. N_COMPONENTES_2D=2
2. gráfico con leyenda de estados

**Prompt sugerido** (copiar al asistente):

```text
X_pca y df_limpio disponibles. ### PEGA AQUÍ ###:
N_COMPONENTES_2D = 2
# scatter X_pca[:,0] vs X_pca[:,1], c=df_limpio['Condition Label']
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
COLOREAR_POR = "Condition Label"
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=df_limpio[COLOREAR_POR], cmap='RdYlGn_r', alpha=0.7, edgecolors='white', linewidth=0.3,
)
ax.set_xlabel(f'PC1 ({var_pc1*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var_pc2*100:.1f}% varianza)')
ax.set_title('Proyección PCA — estados estructurales')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label(COLOREAR_POR)
plt.tight_layout()
plt.show()
n_muestras = len(X_pca)


In [ ]:
# --- Autoevaluación 9 ---
r = verificar_proyeccion_2d(n_muestras, COLOREAR_POR, var_pc1, var_pc2)
resumen_seccion('9 — Proyección 2D', r)


## Pregunta 10 — KMeans y método del codo (elbow)

### 📘 Subpreguntas
- **10.a** ¿Por qué conviene escalar features **antes** de KMeans?
- **10.b** ¿Qué valor de **k** sugiere el gráfico del codo para este edificio instrumentado?

KMeans agrupa lecturas de sensores **sin usar** `Condition Label`; luego compararás con la etiqueta real.


#### ✅ Reflexión 10

Codo sugiere k≈3 (tres estados de daño). KMeans agrupa sin etiqueta; Silhouette moderado es normal en SHM real.


In [ ]:
# --- PRE-ESCRITO: inercia vs k (espacio escalado) ---
K_MIN, K_MAX = 2, 10
inertias = []
for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca_input)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(K_MIN, K_MAX + 1), inertias, 'o-', color='#2980b9', linewidth=2)
ax.set_xlabel('Número de clústeres k')
ax.set_ylabel('Inercia (within-cluster SS)')
ax.set_title('Método del codo — KMeans sobre sensores escalados')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 🤖 Sección 10 — KMeans

**Objetivo:** Método del codo y KMeans con K_OPT.

**Lista de tareas**
1. K_MIN,K_MAX,K_OPT
2. silhouette sil_km
3. gráfico codo + scatter clústeres

**Prompt sugerido** (copiar al asistente):

```text
X_pca_input = X_scaled (o X_pca según celda previa). ### PEGA AQUÍ ###:
K_MIN=2, K_MAX=8, K_OPT=3
# bucle inercias, KMeans, silhouette, plots
```

**Prompt alternativo válido** (misma sección, otros parámetros permitidos):

```text
K_OPT=4 si el codo lo sugiere; sil_km debe calcularse.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
K_OPT = 3
kmeans = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_pca_input)
sil_km = silhouette_score(X_pca_input, labels_km)
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_km, cmap='tab10', alpha=0.7)
ax.set_xlabel(f'PC1 ({var_pc1*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var_pc2*100:.1f}% varianza)')
ax.set_title('KMeans k=3 — agrupación en plano PCA')
plt.colorbar(sc, ax=ax, label='Clúster')
plt.tight_layout()
plt.show()
print(f"Silhouette KMeans = {sil_km:.3f}")


In [ ]:
# --- Autoevaluación 10 ---
r = verificar_kmeans(K_OPT, K_MIN, K_MAX, sil_km)
resumen_seccion('10 — KMeans y codo', r)


## Pregunta 11 — DBSCAN (densidad y ruido)

### 📘 Subpreguntas
- **11.a** ¿Qué significa la etiqueta **-1** en DBSCAN?
- **11.b** ¿Cómo influyen `eps` y `min_samples` en el número de clústeres y puntos ruidosos?

DBSCAN no exige fijar **k**; detecta regiones densas y marca outliers (lecturas atípicas de sensores).


#### ✅ Reflexión 11

DBSCAN marca ruido (-1) en lecturas atípicas; eps pequeño → más ruido, eps grande → pocos clústeres.


In [ ]:
# --- PRE-ESCRITO: sensibilidad rápida a eps (referencia docente) ---
for eps_demo in (0.5, 0.7, 1.0):
    db_demo = DBSCAN(eps=eps_demo, min_samples=8).fit(X_pca_input)
    n_cl = len(set(db_demo.labels_)) - (1 if -1 in db_demo.labels_ else 0)
    n_noise = int((db_demo.labels_ == -1).sum())
    print(f'eps={eps_demo:.1f} → {n_cl} clúster(es), {n_noise} puntos ruido (-1)')


### 🤖 Sección 11 — DBSCAN

**Objetivo:** Clustering por densidad con eps y min_samples.

**Lista de tareas**
1. EPS=0.7, MIN_SAMPLES=8
2. n_clusters_db, n_noise_db, sil_db
3. scatter

**Prompt sugerido** (copiar al asistente):

```text
X_pca_input disponible. ### PEGA AQUÍ ###:
EPS=0.7, MIN_SAMPLES=8
dbscan fit_predict, contar clústeres y ruido (-1), silhouette sin ruido, plot
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
EPS = 0.7
MIN_SAMPLES = 8
dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
labels_db = dbscan.fit_predict(X_pca_input)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = int((labels_db == -1).sum())
mask_db = labels_db != -1
sil_db = silhouette_score(X_pca_input[mask_db], labels_db[mask_db])
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_db, cmap='coolwarm', alpha=0.7)
ax.set_title('DBSCAN — clústeres y puntos ruido')
plt.colorbar(sc, ax=ax, label='Clúster')
plt.tight_layout()
plt.show()
print(f"Clústeres={n_clusters_db}, ruido={n_noise_db}, silhouette={sil_db:.3f}")


In [ ]:
# --- Autoevaluación 11 ---
r = verificar_dbscan(EPS, MIN_SAMPLES, n_clusters_db, n_noise_db, sil_db)
resumen_seccion('11 — DBSCAN', r)


## Pregunta 12 — Comparativa KMeans vs DBSCAN vs etiqueta real

### 📘 Subpreguntas
- **12.a** ¿Qué mide el **índice de Silhouette** (más alto = mejor separación)?
- **12.b** ¿Qué mide el **Adjusted Rand Index (ARI)** frente a `Condition Label`?
- **12.c** ¿Cuál método se alinea mejor con los estados estructurales conocidos?


#### ✅ Reflexión 12

ARI compara con Condition Label solo para evaluar; KMeans suele alinear mejor que DBSCAN con estos parámetros.


In [ ]:
# --- PRE-ESCRITO: etiqueta de referencia (supervisada, solo para comparar) ---
y_true = df_limpio['Condition Label'].values
ari_km = adjusted_rand_score(y_true, labels_km)
ari_db = adjusted_rand_score(y_true[mask_db], labels_db[mask_db]) if mask_db.sum() > 0 else float('nan')
print("Etiquetas reales: 0=normal, 1=daño menor, 2=severo")


### 🤖 Sección 12 — Comparativa

**Objetivo:** ARI entre clústeres y Condition Label.

**Lista de tareas**
1. labels_km y labels_db del notebook
2. ari_km, ari_db

**Prompt sugerido** (copiar al asistente):

```text
### PEGA AQUÍ ###:
y_true = df_limpio['Condition Label'].values
ari_km = adjusted_rand_score(y_true, labels_km)
ari_db = adjusted_rand_score(y_true, labels_db)
print ambos ARI
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
METRICA_PRIORITARIA = 'ari'
comparativa = pd.DataFrame({
    'Método': ['KMeans', 'DBSCAN'],
    'k / clústeres': [K_OPT, n_clusters_db],
    'Silhouette': [sil_km, sil_db],
    'ARI vs Condition Label': [ari_km, ari_db],
    'Puntos ruido (-1)': [0, n_noise_db],
})
display(comparativa.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(2)
axes[0].bar(x, comparativa['Silhouette'], color=['#3498db', '#e67e22'])
axes[0].set_xticks(x); axes[0].set_xticklabels(comparativa['Método'])
axes[0].set_title('Silhouette')
axes[1].bar(x, comparativa['ARI vs Condition Label'], color=['#3498db', '#e67e22'])
axes[1].set_xticks(x); axes[1].set_xticklabels(comparativa['Método'])
axes[1].set_title('ARI vs daño real')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 12 ---
r = verificar_comparativa_clustering(sil_km, sil_db, ari_km, ari_db, METRICA_PRIORITARIA)
resumen_seccion('12 — Comparativa clustering', r)


## Pregunta 13 — Loadings y clasificación (original vs PCA)

### 📘 Subpreguntas
- **13.a** ¿Qué sensor tiene mayor peso (loading) en PC1?
- **13.b** ¿La clasificación con 3 componentes PCA pierde mucho accuracy frente a features originales?


#### ✅ Reflexión 13

Strain lidera loadings PC1. PCA-3 mantiene accuracy similar con menos dimensiones.


In [ ]:
# --- PRE-ESCRITO: matriz de loadings ---
loadings = pd.DataFrame(
    pca_full.components_.T,
    index=FEATURES,
    columns=[f'PC{i+1}' for i in range(len(FEATURES))],
)
feature_pc1 = loadings['PC1'].abs().idxmax()

fig, ax = plt.subplots(figsize=(8, 4))
loadings['PC1'].sort_values().plot(kind='barh', ax=ax, color='#9b59b6')
ax.set_title(f'Loadings PC1 — mayor peso: {feature_pc1}')
ax.set_xlabel('Peso en PC1')
plt.tight_layout()
plt.show()
display(loadings.round(3))


### 🤖 Sección 13 — Clasificación + loadings

**Objetivo:** Random Forest en PCA y top loading PC1.

**Lista de tareas**
1. N_PCA_CLF=3
2. accuracy
3. top feature en loadings PC1

**Prompt sugerido** (copiar al asistente):

```text
Train/test y PCA ya hechos. ### PEGA AQUÍ ###:
N_PCA_CLF = 3
# clasificador RF en componentes, accuracy, loadings PC1 → nombre sensor top
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### Referencia docente (pasa ✅) ###
N_COMPONENTES_ML = 3
TEST_SIZE = 0.2
RANDOM_STATE = 42
X_ml = StandardScaler().fit_transform(df_limpio[FEATURES])
y_ml = df_limpio['Condition Label']
X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_ml,
)
rf_orig = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_orig.fit(X_train, y_train)
acc_orig = accuracy_score(y_test, rf_orig.predict(X_test))
pca_ml = PCA(n_components=N_COMPONENTES_ML, random_state=RANDOM_STATE)
X_train_pca = pca_ml.fit_transform(X_train)
X_test_pca = pca_ml.transform(X_test)
rf_pca = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_pca.fit(X_train_pca, y_train)
acc_pca = accuracy_score(y_test, rf_pca.predict(X_test_pca))
print(f"Accuracy original: {acc_orig:.3f} | PCA-3: {acc_pca:.3f}")


In [ ]:
# --- Autoevaluación 13 ---
r = verificar_loadings(feature_pc1)
r.extend(verificar_clasificacion_pca(acc_orig, acc_pca, N_COMPONENTES_ML))
resumen_seccion('13 — Loadings y clasificación', r)


## Cierre — respuestas sugeridas

- **PC1 / Strain:** la deformación captura daño estructural; domina la primera componente tras escalado.
- **Obra:** visualizar estados en 2D, reducir dimensionalidad antes de clasificadores con muchos sensores.
- **Validación:** inspección visual, histórico de sensores y criterio del ingeniero — PCA no sustituye normativa.
